In [1]:
from utils.config import DATA_DIR, SAVED_MODELS_DIR, RAW_CONFIG
from pathlib import Path
from models.Unet import UNet
from utils.dataset2 import GLORYSDS2, TestSubset
from utils.splitter import test_indices
import torch
import numpy as np

In [2]:
ds_path = DATA_DIR/RAW_CONFIG['datafile']
data = GLORYSDS2(ds_path)
test_idx = test_indices(Path(f"../../{RAW_CONFIG['test_indices']}"))
test_data = TestSubset(data, test_idx, days=True)

Loaded existing test indices from ../../test_indices/2021_test_stratification.pt, test size: 378


In [3]:
print(len(data))
print(len(test_data))
len(test_data[20])

2214
378


3

In [ ]:
from torch.utils.data import DataLoader
import torch
from torch.utils.data._utils.collate import default_collate

def collate_img_lbl_day(batch):
    imgs, lbls, days = zip(*batch)      
    imgs  = default_collate(imgs)      
    lbls  = default_collate(lbls)       
    days  = default_collate(days)       
    return imgs, lbls, days

test_loader = DataLoader(
    test_data,            
    batch_size=1,
    shuffle=True,
    collate_fn=collate_img_lbl_day
)
print(len(next(iter(test_loader))))

ValueError: not enough values to unpack (expected 3, got 2)

In [ ]:
import os
import glob

model_path = SAVED_MODELS_DIR / "training_start_time: 20250520_152727"/"best_model"
model = torch.load(model_path, map_location="cpu", weights_only=False)
model.eval()

unet

In [6]:
outputs = []
num_outputs = 10

with torch.no_grad():
    for i, batch in enumerate(test_loader):
        print(f"Processing batch {i}, len(batch)={len(batch)}")

        if len(batch) == 3:
            images, labels, days = batch
            day_value = days.item() if isinstance(days, torch.Tensor) else days[0]
            print(f"Day: {day_value}")
            preds = model(images)
            outputs.append((preds, labels, days))
        elif len(batch) == 2:
            images, labels = batch
            print("Warning: No day information available, using default day=0")
            preds = model(images)
            # Create a default day value
            default_day = torch.tensor([0])
            outputs.append((preds, labels, default_day))
        else:
            print(f"Warning: Unexpected batch structure with {len(batch)} elements")
        
        if i >= num_outputs:
            break

ValueError: not enough values to unpack (expected 3, got 2)

In [5]:
import matplotlib.pyplot as plt
from utils.handy_plotting_func import plot_var
import numpy as np
import torch.nn.functional as F

# Take fewer samples for better visualization
samples_to_plot = 5
outputs_subset = outputs[:samples_to_plot]

fig, axs = plt.subplots(figsize=(12, 6*len(outputs_subset)), ncols=2, nrows=len(outputs_subset))
for i, output_item in enumerate(outputs_subset):
        pred, label = output_item[:2]
        day = output_item[2].item() 
        pred_np = pred.numpy() if isinstance(pred, torch.Tensor) else pred
        label_np = label.numpy() if isinstance(label, torch.Tensor) else label
        
        pred_unnormalized = data.unnormalize(pred_np, day)
        label_unnormalized = data.unnormalize(label_np, day)
        pred_squeezed = np.squeeze(pred_unnormalized, axis=(0, 1))
        label_squeezed = np.squeeze(label_unnormalized, axis=(0, 1))
            
        plot_var(axs[i, 0], fig, pred_squeezed, f"Predicted Mixed Layer Depth")
        plot_var(axs[i, 1], fig, label_squeezed, f"Actual Mixed Layer Depth")
        pred_tensor = torch.from_numpy(pred_np) if not isinstance(pred, torch.Tensor) else pred
        label_tensor = torch.from_numpy(label_np) if not isinstance(label, torch.Tensor) else label
        normalized_loss = F.mse_loss(pred_tensor, label_tensor)
        axs[i, 0].set_title(f"Predicted Mixed Layer Depth (Day {day})\nMSE Loss: {normalized_loss.item():.8f}")
        axs[i, 1].set_title(f"Actual Mixed Layer Depth (Day {day})\nMSE Loss: {normalized_loss.item():.8f}")
plt.tight_layout()

NameError: name 'outputs' is not defined